In [66]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout, SimpleRNN
from tensorflow.keras.metrics import SparseTopKCategoricalAccuracy
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.activations import leaky_relu
import keras_tuner as kt

In [124]:
X = np.loadtxt('../data/processed/X_8steps_encoded.csv', delimiter=',')
y = np.loadtxt('../data/processed/y_8steps_encoded.csv', delimiter=',')

print('X shape:', X.shape)
print('y shape:', y.shape)

X shape: (211, 132)
y shape: (211,)


In [125]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=5)

print('X Train shape:', X_train.shape)
print('X Test shape:', X_test.shape)
print('y Train shape:', y_train.shape)
print('y Test shape:', y_test.shape)

X Train shape: (189, 132)
X Test shape: (22, 132)
y Train shape: (189,)
y Test shape: (22,)


In [126]:
ann = Sequential([
  Input(shape=(X_train.shape[1],)),
  Dense(512, activation='relu'),
  Dense(256, activation='relu'),
  Dense(128, activation='softmax')
])
ann.summary()

Model: "sequential_16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_45 (Dense)                │ (None, 512)            │        68,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_46 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_47 (Dense)                │ (None, 128)            │        32,896 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 232,320 (907.50 KB)

 Trainable params: 232,320 (907.50 KB)

 Non-trainable params: 0 (0.00 B)

In [127]:
ann.compile(loss='sparse_categorical_crossentropy', optimizer=Adam(), metrics=['accuracy', SparseTopKCategoricalAccuracy(k=5)])
ann.fit(X_train, y_train, epochs=100, validation_split=0.2)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 70ms/step - accuracy: 0.0162 - loss: 4.8374 - sparse_top_k_categorical_accuracy: 0.0448 - val_accuracy: 0.0526 - val_loss: 4.4506 - val_sparse_top_k_categorical_accuracy: 0.2632
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.1157 - loss: 4.2401 - sparse_top_k_categorical_accuracy: 0.2973 - val_accuracy: 0.0789 - val_loss: 4.1113 - val_sparse_top_k_categorical_accuracy: 0.2895
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1144 - loss: 3.7694 - sparse_top_k_categorical_accuracy: 0.3723 - val_accuracy: 0.0789 - val_loss: 3.9367 - val_sparse_top_k_categorical_accuracy: 0.3158
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.1280 - loss: 3.3877 - sparse_top_k_categorical_accuracy: 0.4260 - val_accuracy: 0.1053 - val_loss: 3.9059 - val_sparse_top_k_categorical_accuracy: 0.2632
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2138 - loss: 3.1046 - sparse_top_k_categorical_accuracy: 0.4701 

In [123]:
ann.evaluate(X_test, y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 546ms/step - accuracy: 0.1364 - loss: 7.0292 - sparse_top_k_categorical_accuracy: 0.3182


[7.029241561889648, 0.13636364042758942, 0.3181818127632141]

### Tuning Hyperparameters

In [59]:
def build_model(hp):
  model = Sequential()
  model.add(Input(shape=(129,)))

  # Hidden Layer 1
  model.add(Dense(
    units=hp.Int('units1', min_value=16, max_value=128, step=16),
    activation='relu',
  ))

  # Dropout Layer 1 (Optional)
  if hp.Boolean('use_dropout1_layer'):
    model.add(Dropout(hp.Float('dropout1', min_value=0.1, max_value=0.5, step=0.1)))

  # Hidden Layer 2 (Optional)
  if hp.Boolean('use_hidden_layer2'):
    model.add(Dense(
      units=hp.Int('units2', min_value=16, max_value=128, step=16),
      activation='relu'
    ))

  # Dropout Layer 2 (Optional)
  if hp.Boolean('use_dropout2_layer'):
    model.add(Dropout(hp.Float('dropout2', min_value=0.1, max_value=0.5, step=0.1)))

  # Output Layer
  model.add(Dense(128, activation='softmax'))

  # Optimizer and Learning Rate
  lr = hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])
  optimizer = Adam(learning_rate=lr)

  model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', SparseTopKCategoricalAccuracy(k=5)]
  )

  return model

In [60]:
tuner = kt.RandomSearch(
  build_model,
  objective='val_accuracy',
  max_trials=20,
  executions_per_trial=1,
  directory='../tuning_dir',
  project_name='hero_patch_version'
)

tuner.search(X_train, y_train, epochs=100, validation_split=0.2)

Reloading Tuner from ../tuning_dir\hero_patch_version\tuner0.json


In [61]:
best_model = tuner.get_best_models()[0]
best_hps = tuner.get_best_hyperparameters()[0]

print("Best Hyperparameters:")
print(f"Units1: {best_hps['units1']}")
print(f"Use Dropout1 Layer: {best_hps['use_dropout1_layer']}")
if best_hps['use_dropout1_layer']:
    print(f"Dropout1: {best_hps['dropout1']}")
print(f"Use Second Layer: {best_hps['use_hidden_layer2']}")
if best_hps['use_hidden_layer2']:
    print(f"Units2: {best_hps['units2']}")
print(f"Use Dropout2 Layer: {best_hps['use_dropout2_layer']}")
if best_hps['use_dropout2_layer']:
    print(f"Dropout2: {best_hps['dropout2']}")
print(f"Learning Rate: {best_hps['learning_rate']}")

Best Hyperparameters:
Units1: 112
Use Dropout1 Layer: True
Dropout1: 0.1
Use Second Layer: True
Units2: 80
Use Dropout2 Layer: False
Learning Rate: 0.01


In [62]:
best_model.evaluate(X_test, y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 401ms/step - accuracy: 0.3636 - loss: 4.2502 - sparse_top_k_categorical_accuracy: 0.7273


[4.250194072723389, 0.3636363744735718, 0.7272727489471436]

In [53]:
best_trial = tuner.oracle.get_best_trials(num_trials=1)[0]

print("Best trial ID:", best_trial.trial_id)
print("Best val_accuracy:", best_trial.score)
print("Best hyperparameters:", best_trial.hyperparameters.values)


Best trial ID: 04
Best val_accuracy: 0.5
Best hyperparameters: {'units1': 128, 'use_dropout1_layer': False, 'use_hidden_layer2': True, 'use_dropout2_layer': False, 'learning_rate': 0.001, 'dropout1': 0.30000000000000004, 'units2': 128, 'dropout2': 0.30000000000000004}
